# Class 2 — LP Solver with Tool Use, and Why Tool Design Matters

**Goal.** Show two attempts at the same agent, side by side:

- **Attempt 1 — numeric-coefficient tool.** The tool wants `objective_coefficients=[1.4167, ...]`. The agent has to *compute* those floats — and LLMs are bad at arithmetic. Fragile.
- **Attempt 2 — expression tool.** The tool wants `objective="(5-2-4*12/60-4*8/60-3*5/60)*X_A + ..."`. The agent only *transcribes* numbers from the text; the tool does the arithmetic internally. Reliable.

**You'll learn, step by step:**

1. Why a pure LLM can't solve an LP — it can only describe one.
2. What a **tool** is, and how `@function_tool` auto-generates the JSON schema the model needs.
3. How our first tool design *fails silently* even on the easiest problem we have — because its shape forces the model to do arithmetic in its head.
4. The fix: redesign the tool so the model's job is transcription, not computation. Let the tool parse and evaluate the expressions.
5. The general principle: **the tool's signature is a prompt too.** It dictates what the model has to produce.

Running example throughout: **Problem 1** from `LP_Problems_BUAD449.md` (four products, three machines, weekly profit). Optimal result ≈ **$397.82**.

---


## 0. Setup

Same as Class 1. `.env` at the project root with `OPENAI_API_KEY`; `cvxpy` in the venv.


In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import cvxpy as cp

PROJECT_ROOT = Path.cwd().parent
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — add it to .env at the project root"
client = OpenAI()

MODEL = "gpt-5.4-mini"  # cheap model for demo
print("OK — client ready, cvxpy version:", cp.__version__)

OK — client ready, cvxpy version: 1.8.2


---
## 1. An LLM cannot reliably solve an LP on its own

In Class 1 we got the model to **formulate** an LP — turn a word problem into structured variables, objective, and constraints. That's text manipulation.

Actually **computing the optimum** is a different job. It requires linear-algebra routines (simplex, interior-point). Language models are not calculators; they approximate answers from training data. Watch what happens when we ask the model to solve outright.


In [2]:
PROBLEM_1 = """
A certain plant can manufacture four products A, B, C, and D in any combination.
Each product requires time on each of three machines (in minutes per unit):

    Product   M1   M2   M3
    A         12    8    5
    B          7    9   10
    C          8    4    7
    D         10    0    3

Machine 1, 2, and 3 are available 20, 40, and 10 hours per week respectively.
Material costs are $2 each for products A and B and $1 each for C and D.
All products are competitive and any amounts made may be sold at $5, $4, $5, $5.
Variable labor costs are $4/hour for machines 1 and 2 and $3/hour for machine 3.
Find the best product mix to maximize total weekly profit.
"""

raw = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": (
            "You are an LP solver. Read the problem and return the optimal values "
            "of the decision variables and the optimal objective value. "
            "Give numeric answers only; do not show your work."
        )},
        {"role": "user", "content": PROBLEM_1},
    ],
)
print(raw.output_text)

A = 0  
B = 0  
C = 0  
D = 60  

Maximum profit = 180


Run that a couple of times. The numbers drift from run to run, and often don't satisfy the constraints if you plug them back in. The model *imitates* a solver's output instead of running one. Let's give it a real solver to call.


---
## 2. What a tool is

A **tool** in the Agents SDK is just:

- a **Python function you write**, plus
- a **JSON schema** describing its parameters (auto-generated from the function signature).

The agent framework sends that schema to the model alongside the prompt. The model can then respond with either:

- **plain text** (the normal answer), or
- a **tool call** — a JSON object saying *"please run `solve_linear_program` with these arguments."*

When the SDK sees a tool call, it:

1. Parses the arguments.
2. Runs your function **locally**, in this notebook's process.
3. Sends the return value back as the next turn.
4. Lets the model write a final answer using that result.

```
┌────────┐   prompt    ┌───────┐
│  you   │────────────▶│ model │
└────────┘             └───┬───┘
                           │ "call solve_linear_program(...)"
                           ▼
                      ┌─────────┐
                      │  your   │  (runs locally — your CPU, your process)
                      │  code   │
                      └────┬────┘
                           │ result JSON
                           ▼
                       ┌───────┐
                       │ model │────── final answer to you
                       └───────┘
```

Two things to notice early:

- **The model never runs your code.** It can only *ask* for your code to run.
- **Security.** Whatever your tool function can do (read files, hit APIs, delete things), the model can trigger. Keep tools narrow.


---
## 3. Writing the solver as plain Python

Before involving any agent, write the solver as a normal Python function you can test directly. If it's wrong here, nothing downstream will save you.

**Design choice.** We take the LP in *flat numeric form* (coefficients as lists of floats) rather than Class 1's symbolic expressions. Numbers flow cleanly into CVXPY; strings don't.


In [3]:
from typing import Literal

def _solve_lp_core(
    sense: Literal["maximize", "minimize"],
    variable_names: list[str],
    objective_coefficients: list[float],
    constraint_coefficients: list[list[float]],
    constraint_operators: list[Literal["<=", ">=", "=="]],
    constraint_rhs: list[float],
    constraint_names: list[str],
) -> dict:
    """Plain-Python LP solver. Used directly AND wrapped as a tool below."""
    n = len(variable_names)
    x = cp.Variable(n, nonneg=True)

    obj_expr = sum(c * x[i] for i, c in enumerate(objective_coefficients))
    objective = cp.Maximize(obj_expr) if sense == "maximize" else cp.Minimize(obj_expr)

    constraints = []
    for row, op, rhs in zip(constraint_coefficients, constraint_operators, constraint_rhs):
        lhs = sum(a * x[i] for i, a in enumerate(row))
        if op == "<=":
            constraints.append(lhs <= rhs)
        elif op == ">=":
            constraints.append(lhs >= rhs)
        else:
            constraints.append(lhs == rhs)

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)

    tol = 1e-6
    slacks, binding, shadow_prices = {}, [], {}
    for name, con, row, op, rhs in zip(
        constraint_names, constraints, constraint_coefficients,
        constraint_operators, constraint_rhs,
    ):
        lhs_val = float(sum(a * x.value[i] for i, a in enumerate(row)))
        slack = rhs - lhs_val if op == "<=" else lhs_val - rhs
        slacks[name] = round(slack, 6)
        if abs(slack) < tol:
            binding.append(name)
        shadow_prices[name] = round(float(con.dual_value), 6)

    return {
        "status": prob.status,
        "optimal_value": round(float(prob.value), 4),
        "variable_values": {
            name: round(float(x.value[i]), 6)
            for i, name in enumerate(variable_names)
        },
        "binding_constraints": binding,
        "slacks": slacks,
        "shadow_prices": shadow_prices,
    }

Sanity-check against Problem 1 by hand. Net profit per unit (revenue − material − labor):

- A: `(5−2) − 4·(12+8)/60 − 3·5/60 ≈ 1.4167`
- B: `(4−2) − 4·(7+9)/60 − 3·10/60 ≈ 0.4333`
- C: `(5−1) − 4·(8+4)/60 − 3·7/60  ≈ 2.85`
- D: `(5−1) − 4·(10+0)/60 − 3·3/60 ≈ 3.1833`

Machine capacities in minutes: `1200, 2400, 600`.


In [4]:
manual_result = _solve_lp_core(
    sense="maximize",
    variable_names=["X_A", "X_B", "X_C", "X_D"],
    objective_coefficients=[1.4167, 0.4333, 2.85, 3.1833],
    constraint_coefficients=[
        [12,  7,  8, 10],
        [ 8,  9,  4,  0],
        [ 5, 10,  7,  3],
    ],
    constraint_operators=["<=", "<=", "<="],
    constraint_rhs=[1200, 2400, 600],
    constraint_names=["machine_1", "machine_2", "machine_3"],
)
from pprint import pprint
pprint(manual_result)

{'binding_constraints': ['machine_1', 'machine_3'],
 'optimal_value': 397.8235,
 'shadow_prices': {'machine_1': 0.298546,
                   'machine_2': 0.0,
                   'machine_3': 0.065948},
 'slacks': {'machine_1': 0.0, 'machine_2': 2191.304348, 'machine_3': 0.0},
 'status': 'optimal',
 'variable_values': {'X_A': 0.0, 'X_B': 0.0, 'X_C': 52.173913, 'X_D': 78.26087}}


Optimum ≈ **$397.82**, produce only C and D, machines 1 and 3 binding. Matches the . Our solver is correct — now we can hand it to an agent.


---
## 4. Wrapping it as a tool with `@function_tool`

Adding `@function_tool` to a Python function does three things:

1. Reads the **type hints** and builds a JSON schema.
2. Reads the **docstring** and attaches it as the tool's description (and per-argument descriptions).
3. Registers the function as callable from an `Agent`.

The docstring is not decoration — it's the single most important thing the model reads when deciding *whether* and *how* to call the tool.


In [4]:
from agents import function_tool

@function_tool
def solve_linear_program(
    sense: Literal["maximize", "minimize"],
    variable_names: list[str],
    objective_coefficients: list[float],
    constraint_coefficients: list[list[float]],
    constraint_operators: list[Literal["<=", ">=", "=="]],
    constraint_rhs: list[float],
    constraint_names: list[str],
) -> dict:
    """Solve a continuous linear program with CVXPY (HiGHS solver).

    Use this whenever the user asks you to optimize, maximize, or minimize
    a quantity subject to linear constraints. You must first translate the
    word problem into numeric coefficients before calling.

    All variables are assumed non-negative (>= 0); you do not need to add
    non-negativity constraints yourself.

    Keep units consistent across the objective and every constraint row
    (e.g. if the objective uses hours, don't put minutes in a constraint).

    Args:
        sense: "maximize" or "minimize".
        variable_names: decision-variable names in a fixed order; every
            other coefficient list must align with this order.
        objective_coefficients: coefficient for each variable in the
            objective function.
        constraint_coefficients: 2-D list; one row per constraint, each
            row aligned with variable_names.
        constraint_operators: "<=", ">=", or "==" — one per constraint.
        constraint_rhs: right-hand-side value for each constraint.
        constraint_names: short label for each constraint (for reporting
            which bind at the optimum).

    Returns:
        Dict with status, optimal_value, variable_values, binding_constraints,
        slacks, and shadow_prices.
    """
    return _solve_lp_core(
        sense=sense,
        variable_names=variable_names,
        objective_coefficients=objective_coefficients,
        constraint_coefficients=constraint_coefficients,
        constraint_operators=constraint_operators,
        constraint_rhs=constraint_rhs,
        constraint_names=constraint_names,
    )

print("Tool name       :", solve_linear_program.name)
print("Tool description:", solve_linear_program.description[:110], "...")

Tool name       : solve_linear_program
Tool description: Solve a continuous linear program with CVXPY (HiGHS solver).

Use this whenever the user asks you to optimize, ...


Now look at the schema the SDK will send to the model:


In [8]:
import json
print(json.dumps(solve_linear_program.params_json_schema, indent=2)[:1100], "...")

{
  "properties": {
    "sense": {
      "description": "\"maximize\" or \"minimize\".",
      "enum": [
        "maximize",
        "minimize"
      ],
      "title": "Sense",
      "type": "string"
    },
    "variable_names": {
      "description": "decision-variable names in a fixed order; every\nother coefficient list must align with this order.",
      "items": {
        "type": "string"
      },
      "title": "Variable Names",
      "type": "array"
    },
    "objective_coefficients": {
      "description": "coefficient for each variable in the\nobjective function.",
      "items": {
        "type": "number"
      },
      "title": "Objective Coefficients",
      "type": "array"
    },
    "constraint_coefficients": {
      "description": "2-D list; one row per constraint, each\nrow aligned with variable_names.",
      "items": {
        "items": {
          "type": "number"
        },
        "type": "array"
      },
      "title": "Constraint Coefficients",
      "type": "arr

Your type hints became the schema. `Literal[...]` became `enum`; `list[list[float]]` became nested arrays; every parameter is marked `required` because strict mode is on.

**You never wrote a JSON schema by hand.** That's the whole point.


---
## 5. Pattern A — single agent + tool

Give the `Agent` the tool and the word problem. One call, done. The easiest pattern to write; let's see what it actually produces.


In [5]:
from agents import Agent, Runner

SINGLE_AGENT_INSTRUCTIONS = """
You are an operations-research assistant who solves linear programs for
business users.

When the user gives you an LP word problem:
  1. Identify the decision variables, objective, and constraints.
  2. Convert all units so the objective and every constraint share a
     consistent base.
  3. Call the `solve_linear_program` tool with clean numeric arguments.
  4. Report the result to the user in plain business English.

Never invent numbers. If you cannot translate the problem into numeric
form, say so and stop.
"""

single_agent = Agent(
    name="LPSolverSingle",
    instructions=SINGLE_AGENT_INSTRUCTIONS,
    model=MODEL,
    tools=[solve_linear_program],
)

result_p1 = await Runner.run(single_agent, PROBLEM_1)
print(result_p1.final_output)

The best weekly product mix is:

- **A = 0**
- **B = 0**
- **C = 0**
- **D = 120 units**

### Maximum weekly profit
- **$150**

### What this means
The optimal plan is to use all available **Machine 1** time producing only product **D**. Machines 2 and 3 are left partially unused.

### Notes on the economics
For product D:
- Selling price = **$5**
- Material cost = **$1**
- Machine 1 labor: 10 min = 1/6 hr at $4/hr = **$0.67**
- Machine 3 labor: 3 min = 1/20 hr at $3/hr = **$0.15**

So profit per unit of D is about **$3.18**, and because Machine 1 is the limiting resource, D gives the best use of that capacity.

If you want, I can also show the full profit calculation for all four products.


**Compare the number above to the  optimum, ~$397.82.** Run the cell above a few times. You'll see it land far from $397.82 on at least some runs — and crucially, the business-English summary reads confidently either way.

The steps the SDK ran to produce that answer are in `result_p1.new_items`:


In [6]:
from agents import ToolCallItem, ToolCallOutputItem, MessageOutputItem

for i, item in enumerate(result_p1.new_items, 1):
    kind = type(item).__name__
    print(f"[{i}] {kind}")
    if isinstance(item, ToolCallItem):
        print(f"    tool = {item.raw_item.name}")
        print(f"    args = {item.raw_item.arguments[:200]}...")
    elif isinstance(item, ToolCallOutputItem):
        out = item.output if isinstance(item.output, str) else str(item.output)
        print(f"    output = {out[:200]}...")
    elif isinstance(item, MessageOutputItem):
        text = "".join(c.text for c in item.raw_item.content if hasattr(c, "text"))
        print(f"    text = {text[:200]}...")
    print()

[1] ToolCallItem
    tool = solve_linear_program
    args = {"sense":"maximize","variable_names":["A","B","C","D"],"objective_coefficients":[-1.0666666667,0.4333333333,0.2333333333,1.25],"constraint_coefficients":[[12,7,8,10],[8,9,4,0],[5,10,7,3]],"constraint_...

[2] ToolCallOutputItem
    output = {'status': 'optimal', 'optimal_value': 150.0, 'variable_values': {'A': 0.0, 'B': 0.0, 'C': 0.0, 'D': 120.0}, 'binding_constraints': ['M1_hours'], 'slacks': {'M1_hours': 0.0, 'M2_hours': 2400.0, 'M3_ho...

[3] MessageOutputItem
    text = The best weekly product mix is:

- **A = 0**
- **B = 0**
- **C = 0**
- **D = 120 units**

### Maximum weekly profit
- **$150**

### What this means
The optimal plan is to use all available **Machine 1...



You can read the exact `args` JSON the model sent to the tool, and the `output` the tool returned. Since we know the  optimum is ~$397.82, any other number in `output` means the model-generated arguments were wrong.

Let's pull those arguments out cleanly and see what went wrong.


---
## 6. Diagnosing what went wrong

Problem 1 asks the model to do three separate mental conversions before it can produce tool arguments:

- compute **per-unit profit** for each product (revenue − material − labor, with labor split across three machines at two different hourly rates),
- convert machine capacity from **hours to minutes** (or minutes to hours) so the objective and the constraints share one base, and
- fill in objective coefficients for **four variables** with **zero arithmetic mistakes**.

Any one of those three can break a single-shot call. The only evidence we have is the tool-call arguments. Let's print them.


Inspect the arguments the model sent to the solver — this is the useful evidence:


In [9]:
tool_calls = [it for it in result_p1.new_items if isinstance(it, ToolCallItem)]
tool_outs  = [it for it in result_p1.new_items if isinstance(it, ToolCallOutputItem)]

print(f"tool calls made: {len(tool_calls)}\n")

for tc in tool_calls:
    args = json.loads(tc.raw_item.arguments)
    print("variable_names         :", args.get("variable_names"))
    print("objective_coefficients :", args.get("objective_coefficients"))
    print("constraint_coefficients:")
    for name, row, op, rhs in zip(
        args.get("constraint_names", []),
        args.get("constraint_coefficients", []),
        args.get("constraint_operators", []),
        args.get("constraint_rhs", []),
    ):
        print(f"  [{name}] {row} {op} {rhs}")

if tool_outs:
    out = tool_outs[0].output
    out_str = out if isinstance(out, str) else str(out)
    print("\ntool return (first 300 chars):", out_str[:300])

tool calls made: 1

variable_names         : ['A', 'B', 'C', 'D']
objective_coefficients : [-1.0666666667, 0.4333333333, 0.2333333333, 1.25]
constraint_coefficients:
  [M1_hours] [12, 7, 8, 10] <= 1200
  [M2_hours] [8, 9, 4, 0] <= 2400
  [M3_hours] [5, 10, 7, 3] <= 600

tool return (first 300 chars): {'status': 'optimal', 'optimal_value': 150.0, 'variable_values': {'A': 0.0, 'B': 0.0, 'C': 0.0, 'D': 120.0}, 'binding_constraints': ['M1_hours'], 'slacks': {'M1_hours': 0.0, 'M2_hours': 2400.0, 'M3_hours': 240.0}, 'shadow_prices': {'M1_hours': 0.125, 'M2_hours': 0.0, 'M3_hours': 0.0}}


Compare what the model sent with the reference from Section 3:

- **objective coefficients should be roughly** `[1.4167, 0.4333, 2.85, 3.1833]` — each is a four-term arithmetic chain.
- **constraint_coefficients** should be `[12,7,8,10]`, `[8,9,4,0]`, `[5,10,7,3]` with RHS `[1200, 2400, 600]` in minutes (or divide all by 60 for hours — but consistently).
- **sense** should be `"maximize"`.

Run the cell above a few times. You'll typically see **some combination** of:

- labor cost missing from the objective,
- labor rates counted on the wrong machine,
- constraints in hours but coefficients in minutes (or vice versa),
- off-by-a-factor-of-60 RHS values.

Even when the model gets it right on a run, **you can't tell from the business summary.**

**Diagnosis — and it's not what you might expect.** The usual reaction is "the model made mistakes; we need a better prompt or a smarter model." But look again at what the model had to do: compute `5 - 2 - 4*12/60 - 4*8/60 - 3*5/60` in its head and put `1.4167` into a JSON field. That's **chained arithmetic** — the single thing LLMs are worst at.

The model doesn't have trouble understanding the word problem. It has trouble doing the four multiplications. We asked it to, because our **tool signature** requires it:

```python
objective_coefficients: list[float]    # ← forces the model to compute
```

**The fix is not a better prompt. It's a better tool.** If the tool accepted algebraic expressions instead of floats, the model could just *transcribe* the numbers from the text and the tool itself would do the arithmetic — deterministically, correctly.

That is Section 7.


---
## 7. Attempt 2 — redesign the tool to take expressions

Same agent architecture (one LLM, one tool). The only thing we change is the **tool's signature**:

```
BEFORE (numeric)                                  AFTER (expressions)
─────────────────────────────                     ──────────────────────────────────────
objective_coefficients:  list[float]              objective:   str
constraint_coefficients: list[list[float]]        constraints: list[str]
constraint_rhs:          list[float]
       ▲                                                 ▲
       │ model must compute 1.4167, 0.4333, ...          │ model only transcribes:
       │ (arithmetic in its head)                        │ "(5-2-4*12/60-4*8/60-3*5/60)*X_A + ..."
```

What the new tool does internally:

1. Create a CVXPY `Variable` for each name in `variable_names`, put them in a dictionary.
2. `eval` the objective string against that dictionary — CVXPY variables overload `+`, `-`, `*`, `/`, `<=`, `>=`, `==`, so the result is a real CVXPY expression.
3. `eval` each constraint string the same way — each becomes a CVXPY `Constraint` object.
4. Hand the whole thing to CVXPY and return the result.

The arithmetic is still done — but it's done inside the tool, by Python, deterministically. Zero model arithmetic risk.

**Safety note.** Calling `eval` on LLM output is dangerous in general. We do it inside a **restricted namespace** that contains only the CVXPY variables and `__builtins__={}` — no file access, no imports, no function calls. Any stray name or function call raises a `NameError` before it can do anything. For classroom/prototype use this is fine; in production you'd use a small expression parser (e.g. `ast.parse` with a whitelist of node types, or `sympy.sympify`).


---
## 8. Building the expression-based tool

Same pattern as Section 3/4 — plain Python function first, then wrap with `@function_tool`.


In [10]:
def _solve_lp_from_exprs(
    sense: Literal["maximize", "minimize"],
    variable_names: list[str],
    objective: str,
    constraints: list[str],
    constraint_names: list[str],
) -> dict:
    """Solve an LP written as algebraic-expression strings.

    The expressions may reference any name in `variable_names` and use
    +, -, *, /, (), and the comparison operators <=, >=, ==. The tool
    creates one CVXPY Variable per name, evaluates the expressions
    inside a restricted namespace (no builtins, no outside scope), and
    hands the resulting CVXPY problem to HiGHS.
    """
    cvx_vars = {name: cp.Variable(nonneg=True) for name in variable_names}
    safe_globals = {"__builtins__": {}}

    try:
        obj_expr = eval(objective, safe_globals, cvx_vars)
    except Exception as e:
        raise ValueError(f"could not parse objective {objective!r}: {e}") from e

    prob_obj = cp.Maximize(obj_expr) if sense == "maximize" else cp.Minimize(obj_expr)

    cvx_cons = []
    for c_str in constraints:
        try:
            cvx_cons.append(eval(c_str, safe_globals, cvx_vars))
        except Exception as e:
            raise ValueError(f"could not parse constraint {c_str!r}: {e}") from e

    prob = cp.Problem(prob_obj, cvx_cons)
    prob.solve(solver=cp.HIGHS)

    tol = 1e-5
    slacks, binding, shadow = {}, [], {}
    for name, con in zip(constraint_names, cvx_cons):
        try:
            lhs_val = float(con.args[0].value)
            rhs_val = float(con.args[1].value)
            gap = lhs_val - rhs_val
            slacks[name] = round(gap, 6)
            if abs(gap) < tol:
                binding.append(name)
        except Exception:
            slacks[name] = None
        shadow[name] = (
            round(float(con.dual_value), 6) if con.dual_value is not None else None
        )

    return {
        "status": prob.status,
        "optimal_value": round(float(prob.value), 4),
        "variable_values": {
            name: round(float(cvx_vars[name].value), 6) for name in variable_names
        },
        "binding_constraints": binding,
        "slacks": slacks,
        "shadow_prices": shadow,
    }

# Smoke-test with the reference expressions for Problem 1:
_smoke = _solve_lp_from_exprs(
    sense="maximize",
    variable_names=["X_A", "X_B", "X_C", "X_D"],
    objective=(
        "(5-2-4*12/60-4*8/60-3*5/60)*X_A + (4-2-4*7/60-4*9/60-3*10/60)*X_B + "
        "(5-1-4*8/60-4*4/60-3*7/60)*X_C + (5-1-4*10/60-4*0/60-3*3/60)*X_D"
    ),
    constraints=[
        "12*X_A + 7*X_B + 8*X_C + 10*X_D <= 20*60",
        "8*X_A + 9*X_B + 4*X_C + 0*X_D <= 40*60",
        "5*X_A + 10*X_B + 7*X_C + 3*X_D <= 10*60",
    ],
    constraint_names=["machine_1", "machine_2", "machine_3"],
)
print("smoke test optimum:", _smoke["optimal_value"], "(expected ≈ 397.82)")

smoke test optimum: 397.8261 (expected ≈ 397.82)


Smoke test matches $397.82. Now wrap it with `@function_tool` so an agent can call it. Notice how much simpler the signature is — four fields instead of seven, and no 2-D arrays:


In [11]:
@function_tool
def solve_lp_from_expressions(
    sense: Literal["maximize", "minimize"],
    variable_names: list[str],
    objective: str,
    constraints: list[str],
    constraint_names: list[str],
) -> dict:
    """Solve a continuous linear program given as algebraic expressions.

    Do NOT precompute coefficients. Write out each expression using only
    raw numbers from the problem and the operators +, -, *, /, parentheses.
    Example for a 4-product profit objective (time in minutes, rates per hour):

        objective = "(5-2-4*12/60-4*8/60-3*5/60)*X_A + (4-2-...)*X_B + ..."

    Each constraint is ONE string that includes its comparison operator:

        constraints = [
            "12*X_A + 7*X_B + 8*X_C + 10*X_D <= 20*60",
            ...
        ]

    All variables are non-negative by default; you do NOT need to include
    x >= 0 constraints.

    Args:
        sense: "maximize" or "minimize".
        variable_names: list of decision-variable names used in the
            expressions (e.g. ["X_A", "X_B"]). Names must be valid Python
            identifiers.
        objective: algebraic expression for the objective function.
        constraints: list of algebraic inequalities or equalities, one
            string per constraint.
        constraint_names: short label for each constraint (same length
            as constraints).

    Returns:
        Dict with status, optimal_value, variable_values, binding_constraints,
        slacks, and shadow_prices.
    """
    return _solve_lp_from_exprs(
        sense=sense,
        variable_names=variable_names,
        objective=objective,
        constraints=constraints,
        constraint_names=constraint_names,
    )

print("Tool name       :", solve_lp_from_expressions.name)
print("Tool description:", solve_lp_from_expressions.description[:110], "...")
print()
print("Params JSON schema (compare to the numeric tool in Section 4):")
print(json.dumps(solve_lp_from_expressions.params_json_schema, indent=2)[:700], "...")

Tool name       : solve_lp_from_expressions
Tool description: Solve a continuous linear program given as algebraic expressions.

Do NOT precompute coefficients. Write out e ...

Params JSON schema (compare to the numeric tool in Section 4):
{
  "properties": {
    "sense": {
      "description": "\"maximize\" or \"minimize\".",
      "enum": [
        "maximize",
        "minimize"
      ],
      "title": "Sense",
      "type": "string"
    },
    "variable_names": {
      "description": "list of decision-variable names used in the\nexpressions (e.g. [\"X_A\", \"X_B\"]). Names must be valid Python\nidentifiers.",
      "items": {
        "type": "string"
      },
      "title": "Variable Names",
      "type": "array"
    },
    "objective": {
      "description": "algebraic expression for the objective function.",
      "title": "Objective",
      "type": "string"
    },
    "constraints": {
      "description": "list of algebr ...


---
## 9. Running Attempt 2 on Problem 1

Same `Agent` class, same model, same word problem. Only the tool is different — it now takes expressions instead of numeric coefficients.


In [12]:
SMART_AGENT_INSTRUCTIONS = """
You are an operations-research assistant who solves linear programs for
business users by calling the `solve_lp_from_expressions` tool.

CRITICAL: Do not compute anything yourself. Do not write floats like
1.4167 or 0.4333. Your job is transcription — write algebraic expressions
containing ONLY the raw numbers stated in the problem plus + - * / ( ).

Example objective for a 4-product / 3-machine profit problem:
    "(5-2-4*12/60-4*8/60-3*5/60)*X_A + (4-2-...)*X_B + ..."
Example constraint:
    "12*X_A + 7*X_B + 8*X_C + 10*X_D <= 20*60"

Keep units consistent across the objective and every constraint (pick
minutes everywhere or hours everywhere). Variables are non-negative;
do not add x >= 0 rows. After the tool returns, write the result in
plain business English.
"""

smart_agent = Agent(
    name="LPSolverSmart",
    instructions=SMART_AGENT_INSTRUCTIONS,
    model=MODEL,
    tools=[solve_lp_from_expressions],
)

result_smart = await Runner.run(smart_agent, PROBLEM_1)
print(result_smart.final_output)

The best weekly product mix is:

- Product A: 0 units
- Product B: 0 units
- Product C: 52.173913 units
- Product D: 78.26087 units

Maximum weekly profit: **$397.8261**

In business terms, the optimal plan is to produce only products **C** and **D**, using all of Machine 1 and Machine 3 capacity. Machine 2 is not a limiting factor in the optimal mix.


Now look at the tool call the model produced. This is the real evidence the redesign worked:


In [13]:
for tc in (it for it in result_smart.new_items if isinstance(it, ToolCallItem)):
    args = json.loads(tc.raw_item.arguments)
    print("sense          :", args.get("sense"))
    print("variable_names :", args.get("variable_names"))
    print()
    print("objective:")
    print(" ", args.get("objective"))
    print()
    print("constraints:")
    for name, c in zip(args.get("constraint_names", []), args.get("constraints", [])):
        print(f"  [{name}] {c}")

for to in (it for it in result_smart.new_items if isinstance(it, ToolCallOutputItem)):
    out_str = to.output if isinstance(to.output, str) else str(to.output)
    print("\ntool return (first 350 chars):", out_str[:350])

sense          : maximize
variable_names : ['X_A', 'X_B', 'X_C', 'X_D']

objective:
  (5-2-(4/60)*(12+8)+( - ) )

constraints:
  [M1] 12*X_A + 7*X_B + 8*X_C + 10*X_D <= 20*60
  [M2] 8*X_A + 9*X_B + 4*X_C + 0*X_D <= 40*60
  [M3] 5*X_A + 10*X_B + 7*X_C + 3*X_D <= 10*60
sense          : maximize
variable_names : ['X_A', 'X_B', 'X_C', 'X_D']

objective:
  (5-2-(12*4/60+8*4/60+5*3/60))*X_A + (4-2-(7*4/60+9*4/60+10*3/60))*X_B + (5-1-(8*4/60+4*4/60+7*3/60))*X_C + (5-1-(10*4/60+0*4/60+3*3/60))*X_D

constraints:
  [M1] 12*X_A + 7*X_B + 8*X_C + 10*X_D <= 20*60
  [M2] 8*X_A + 9*X_B + 4*X_C + 0*X_D <= 40*60
  [M3] 5*X_A + 10*X_B + 7*X_C + 3*X_D <= 10*60

tool return (first 350 chars): An error occurred while running the tool. Please try again. Error: could not parse objective '(5-2-(4/60)*(12+8)+( - ) )': invalid syntax (<string>, line 1)

tool return (first 350 chars): {'status': 'optimal', 'optimal_value': 397.8261, 'variable_values': {'X_A': 0.0, 'X_B': 0.0, 'X_C': 52.173913, 'X_D': 78.26087}, 

### Compare Attempt 1 vs Attempt 2 on Problem 1

- **Attempt 1 (numeric tool):** model had to compute `1.4167`, `0.4333`, `2.85`, `3.1833` in its head and fill in a flat numeric array. Chained arithmetic is exactly what LLMs fail at, so the optimum drifted and you couldn't tell from the business summary when it was wrong.
- **Attempt 2 (expression tool):** model only transcribed the raw numbers from the text (`5`, `2`, `4`, `12`, `60`, …) into a single algebraic expression. The tool did the arithmetic. The optimum is ~$397.82 reliably.

**Same model. Same `@function_tool` decorator. Same CVXPY underneath.** The only thing we changed was the **tool's JSON schema** — moving from `list[float]` fields to one `str` field. The schema is what dictates the model's job description, and moving the arithmetic from the model to the tool is what made the agent work.

**Principle to take away:** *the tool's signature is a prompt too.* When designing a tool, ask yourself what the model will have to compute to satisfy it. Whenever the answer is "chained arithmetic," redesign the tool.


---
## 10. Cost

Token accounting for Attempt 1 vs Attempt 2 on Problem 1. They should be roughly similar — the difference in reliability comes from tool design, not from spending more tokens.


In [14]:
PRICES = {
    "gpt-5.4":      {"in": 2.5, "out": 15.00},
    "gpt-5.4-mini": {"in": 0.75, "out": 4.5},
    "gpt-5.4-nano": {"in": 0.2, "out":  1.25},
}

def cost_usd(model: str, usage) -> float:
    p = PRICES[model]
    return usage.input_tokens / 1e6 * p["in"] + usage.output_tokens / 1e6 * p["out"]

# Re-run both agents on Problem 1 for clean usage numbers.
fresh_attempt1 = await Runner.run(single_agent, PROBLEM_1)
fresh_attempt2 = await Runner.run(smart_agent,  PROBLEM_1)

u1 = fresh_attempt1.context_wrapper.usage
u2 = fresh_attempt2.context_wrapper.usage

print("Attempt 1 (numeric tool):")
print(f"  input  tokens: {u1.input_tokens}")
print(f"  output tokens: {u1.output_tokens}")
print(f"  cost         : ${cost_usd(MODEL, u1):.6f}")

print("\nAttempt 2 (expression tool):")
print(f"  input  tokens: {u2.input_tokens}")
print(f"  output tokens: {u2.output_tokens}")
print(f"  cost         : ${cost_usd(MODEL, u2):.6f}")

Attempt 1 (numeric tool):
  input  tokens: 1600
  output tokens: 258
  cost         : $0.002361

Attempt 2 (expression tool):
  input  tokens: 1893
  output tokens: 344
  cost         : $0.002968


A few things you'll typically see:

- Total token cost is very similar between the two attempts — both make one tool call and one final message.
- Attempt 2's output tokens can actually be *smaller* because the model doesn't have to emit a long flat numeric array; one short algebraic string is more compact than four floats plus a 3×4 matrix.
- The reliability gain from Attempt 2 is essentially **free** — we didn't spend more on a bigger model or a longer prompt; we just redesigned the tool.

The general rule stays: **use an LLM only where judgment is required; use code for arithmetic.** The tool's JSON schema is where you enforce that separation.


---
## Practice — try `smart_agent` yourself

Stay in **this** notebook for these cells — no new file, no submission.

Pick **one** problem from `LP_Problems_BUAD449.md` other than Problem 1. Good candidates, in rough order of translation difficulty:

- **Problem 2** — tiered pricing with overtime hours (11 variables).
- **Problem 3** — Buster Sod's farm (has an equality alfalfa-balance constraint, unit conversion between tons and acres).
- **Problem 4** — GreenBridge portfolio (linearized ratio constraints — easy to get a sign wrong).
- **Problem 5** — heating-oil blending (proportional blend specs that need linearization).

Work through these steps in a few new cells below:

1. Put the problem text (drop the  "Formulation" section) into a Python string called `PROBLEM`.
2. Run the smart agent:

   ```python
   result_practice = await Runner.run(smart_agent, PROBLEM)
   print(result_practice.final_output)
   ```

3. **Inspect the tool call arguments before trusting the answer.** Reuse the cell from Section 9 but point it at `result_practice.new_items`. Each expression should contain only raw numbers the problem actually states (plus `+ - * / ( )` and variable names). If you spot a pre-computed float, the model cheated — tighten the instructions or re-run.
4. Compare the optimal value to the . If they mismatch, read the tool-call arguments — the error is visible there, not in the solver.

The whole point of the exercise is the **inspection in step 3** — the algebraic tool call is the auditable intermediate that makes Attempt 2 reliable.
